# Process to make the 1st order consensus

We want to make a consensus training dataset, by running extractions on the 1000 sample images using the models fine-tuned on the fake data, and then making a set of consensus transcriptions - where we judge the accuracy of each transcription by whether we get agreement between the models. We will then do further fine-tuneing on this consensus dataset.


## 1. Model checkpoints to use

We will use the 5 model checkpoints fine-tuned on the fake data

- Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260601-154723/HuggingFaceTB--SmolVLM2-2.2B-Instruct
- Daily_rainfall_sample/outputs/checkpoints/granite4-20260601-121821/ibm-granite--granite-vision-4.1-4b
- Daily_rainfall_sample/outputs/checkpoints/gemma3-20260601-121832/google--gemma-3-4b-it
- Daily_rainfall_sample/outputs/checkpoints/gemma4-20260601-121840/google--gemma-4-E4B-it
- Daily_rainfall_sample/outputs/checkpoints/ministral-20260601-121858/mistralai--Mistral-Small-3.1-24B-Instruct-2503

We will use the set of 1000 images drawn as a random sample from the full image set.

- documents/DailyRainfall_UK/consensus_1000/images


## 2. Run the extractions

Each command submits one extraction job to Azure. Run them in the terminal. They will complete in about 2.5 hours (+queuing time).

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260601-154723/HuggingFaceTB--SmolVLM2-2.2B-Instruct --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/granite4-20260601-121821/ibm-granite--granite-vision-4.1-4b --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/gemma3-20260601-121832/google--gemma-3-4b-it --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/gemma4-20260601-121840/google--gemma-4-E4B-it --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions --total-shards 1 --batch-size 50 --extraction-registry outputs/extraction_registry.json extract 
```

```bash
bash /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/scripts/aml_submit.sh --checkpoint Daily_rainfall_sample/outputs/checkpoints/ministral-20260601-121858/mistralai--Mistral-Small-3.1-24B-Instruct-2503 --images-path documents/DailyRainfall_UK/consensus_1000/images --transcriptions-path documents/DailyRainfall_UK/consensus_1000/transcriptions --total-shards 4 --batch-size 15 --extraction-registry outputs/extraction_registry.json extract 
```


## 4. Download the extractions from Azure

Get the extraction run_names from the extraction registry. In this case, they were:

- SmolVLM: 20260605-203055
- Granite4: 20260605-205209
- Gemma3: 20260605-205255
- Gemma4: 20260605-205328
- Ministral: 20260605-205414

Then download them in turn

```bash
bash scripts/aml_download.sh --run-name 20260605-203055
bash scripts/aml_download.sh --run-name 20260605-205209
bash scripts/aml_download.sh --run-name 20260605-205255
bash scripts/aml_download.sh --run-name 20260605-205328
bash scripts/aml_download.sh --run-name 20260605-205414
```




## 5. Make consensus from the downloaded extractions

Looks for cases where multiple extractions agree and flags those sections as reliable.

```bash
bash scripts/run_consensus_pipeline.sh --dataset-root outputs/consensus_dataset_1000 --variant-name consensus_1st_order --threshold 3 --null-threshold 5 --precision 3 --extraction-dir outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260605-203055 --extraction-dir outputs/extractions/ibm-granite--granite-vision-4.1-4b/20260605-205209 --extraction-dir outputs/extractions/google--gemma-3-4b-it/20260605-205255 --extraction-dir outputs/extractions/google--gemma-4-E4B-it/20260605-205328 --extraction-dir outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260605-205414 --validate --validation-sample-denominator 20
```

This will produce a consensus in ```outputs/consensus_dataset_1000/consensus_1st_order``` with transcriptions (ready to use for further fine-tuning), summary and validation figures for 1/20 of the cases.